# OfferBloom Interview Question Pipeline

Three sources:
1. **Reddit** (PRAW) — fresh crowd reports from r/cscareerquestions, r/leetcode, etc.
2. **GitHub** — curated markdown repos (system design, ML, DS interview lists)
3. **Stack Exchange API** — concept questions (SQL, Python, ML, system design)

Output: normalized JSONL at `output/questions.jsonl`

Schema per record:
```json
{
  "id": "<source>_<hash>",
  "source": "reddit|github|stackexchange",
  "company": "google|amazon|...|unknown",
  "role": "swe|pm|ds|ml|fde|unknown",
  "topic": "behavioral|system_design|coding|ml|sql|...",
  "question": "...",
  "answer": "...",
  "url": "...",
  "scraped_at": "2025-..."
}
```

## 0. Setup

In [6]:
# Install deps (run once)
# !pip install praw requests gitpython tqdm python-dotenv

In [3]:
import os
import re
import json
import hashlib
import time
import datetime
import pathlib
from typing import Optional

import requests
from tqdm.notebook import tqdm

OUTPUT_DIR = pathlib.Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
JSONL_PATH = OUTPUT_DIR / "questions.jsonl"

# ── helpers ──────────────────────────────────────────────────────────────

def make_id(source: str, text: str) -> str:
    return source + "_" + hashlib.md5(text.encode()).hexdigest()[:10]

def now_iso() -> str:
    return datetime.datetime.utcnow().isoformat() + "Z"

def append_records(records: list[dict]):
    with open(JSONL_PATH, "a", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  → wrote {len(records)} records")

# ── classifiers (keyword-based, fast, good-enough for preload) ────────────

COMPANIES = [
    "google", "amazon", "meta", "apple", "microsoft", "netflix", "uber",
    "airbnb", "stripe", "openai", "anthropic", "scale", "databricks",
    "palantir", "salesforce", "linkedin", "twitter", "snap", "lyft",
    "doordash", "instacart", "coinbase", "robinhood", "figma", "notion",
]

ROLE_KEYWORDS = {
    "pm": ["product manager", " pm ", "product sense", "product design"],
    "swe": ["software engineer", " swe ", "coding round", "leetcode", "system design"],
    "ds": ["data scientist", " ds ", "statistics", "a/b test", "sql"],
    "ml": ["machine learning", " ml ", "model", "neural", "transformer", "nlp"],
    "fde": ["forward deployed", " fde ", "solutions engineer", "field engineer"],
}

TOPIC_KEYWORDS = {
    "behavioral": ["tell me about", "describe a time", "conflict", "leadership", "why "],
    "system_design": ["design a", "architect", "scalab", "microservice", "api design"],
    "coding": ["leetcode", "algorithm", "data structure", "time complexity", "big o"],
    "ml": ["model", "training", "overfitting", "gradient", "feature", "neural"],
    "sql": ["select", "join", "query", "database", "schema"],
    "product": ["product sense", "metrics", "launch", "north star", "prioritiz"],
    "statistics": ["p-value", "hypothesis", "distribution", "confidence interval", "bias variance"],
}

def classify_company(text: str) -> str:
    t = text.lower()
    for c in COMPANIES:
        if c in t:
            return c
    return "unknown"

def classify_role(text: str) -> str:
    t = text.lower()
    for role, kws in ROLE_KEYWORDS.items():
        if any(kw in t for kw in kws):
            return role
    return "unknown"

def classify_topic(text: str) -> str:
    t = text.lower()
    for topic, kws in TOPIC_KEYWORDS.items():
        if any(kw in t for kw in kws):
            return topic
    return "general"

print("Setup done.")

Setup done.


In [7]:
pip install kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [kagglehub]

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
# Install dependencies as needed:

import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "data_pipeline/output/questions.json"

# Load the latest version
df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "aryan208/hr-interview-questions-and-ideal-answers", file_path)

print("First 5 records:", df.head())

/var/folders/4s/5_6fptlx1nn617t00pcg6kq40000gn/T/ipykernel_93641/373862127.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "aryan208/hr-interview-questions-and-ideal-answers", file_path)


KaggleApiHTTPError: 404 Client Error.

Resource not found at URL: https://kaggle.com/datasets/aryan208/hr-interview-questions-and-ideal-answers/versions/1
Please make sure you specified the correct resource identifiers.

In [19]:
import pandas as pd                                                                  
                                                                                     
CACHE_PATH = "/Users/massacoulibaly/.cache/kagglehub/datasets/aryan208/hr-interview-questions-and-ideal-answers/versions/1/hr_interview_questions_dataset.json"         
                                                                                       
df = pd.read_json(CACHE_PATH)                                                      
print(f"Loaded {len(df)} records")                                                 
print(df.head()) 

Loaded 2500000 records
                                            question             category  \
0  Tell me about a time you had to learn somethin...         Adaptability   
1  Describe a time you handled a difficult situat...  Conflict Resolution   
2              Where do you see yourself in 5 years?         Career Goals   
3  Tell me about a conflict you had with a cowork...  Conflict Resolution   
4  What skills do you hope to develop in your nex...         Career Goals   

                  role experience difficulty  source_type  \
0      DevOps Engineer    fresher       Easy   Open-Ended   
1      Product Manager    2 years       Hard   Behavioral   
2       Data Scientist    2 years     Medium   Open-Ended   
3  Marketing Associate    4 years       Hard   STAR-Based   
4  Marketing Associate     1 year       Easy  Situational   

                                        ideal_answer              keywords  
0  I'm always eager to learn and embrace change a...    [flexible, cha

## 1. Reddit via PRAW

Get credentials at https://www.reddit.com/prefs/apps → create app → script type.

Set these env vars or paste values below:
```
REDDIT_CLIENT_ID=...
REDDIT_CLIENT_SECRET=...
REDDIT_USER_AGENT=offerbloom_scraper/0.1
```

In [ ]:
import praw

REDDIT_CLIENT_ID     = os.getenv("REDDIT_CLIENT_ID", "YOUR_CLIENT_ID")
REDDIT_CLIENT_SECRET = os.getenv("REDDIT_CLIENT_SECRET", "YOUR_CLIENT_SECRET")
REDDIT_USER_AGENT    = os.getenv("REDDIT_USER_AGENT", "offerbloom_scraper/0.1")

reddit = praw.Reddit(
    client_id=REDDIT_CLIENT_ID,
    client_secret=REDDIT_CLIENT_SECRET,
    user_agent=REDDIT_USER_AGENT,
)

SUBREDDITS = [
    "cscareerquestions",
    "ExperiencedDevs",
    "leetcode",
    "datascience",
    "MachineLearning",
    "ProductManagement",
]

SEARCH_QUERIES = [
    "interview experience",
    "onsite interview",
    "interview questions asked",
    "passed interview",
    "failed interview",
    "loop experience",
]

POSTS_PER_QUERY = 25  # bump to 100 for full run


def extract_questions_from_text(text: str) -> list[str]:
    """Pull sentences ending in ? from post body."""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip().endswith("?") and len(s) > 20]


def scrape_reddit() -> list[dict]:
    records = []
    seen_ids = set()

    for sub_name in tqdm(SUBREDDITS, desc="Subreddits"):
        sub = reddit.subreddit(sub_name)
        for query in SEARCH_QUERIES:
            try:
                posts = sub.search(query, sort="new", limit=POSTS_PER_QUERY)
                for post in posts:
                    if post.id in seen_ids:
                        continue
                    seen_ids.add(post.id)

                    body = post.selftext or ""
                    full_text = post.title + " " + body
                    questions = extract_questions_from_text(full_text)

                    # Also treat the whole post as one question entry
                    if not questions and len(body) > 100:
                        questions = [post.title]

                    # Grab top comment as answer proxy
                    post.comments.replace_more(limit=0)
                    top_comments = sorted(
                        [c for c in post.comments.list() if hasattr(c, 'score')],
                        key=lambda c: c.score,
                        reverse=True,
                    )
                    answer = top_comments[0].body if top_comments else ""

                    for q in questions:
                        rec = {
                            "id": make_id("reddit", q),
                            "source": "reddit",
                            "company": classify_company(full_text),
                            "role": classify_role(full_text),
                            "topic": classify_topic(q),
                            "question": q,
                            "answer": answer[:2000],  # cap length
                            "url": f"https://reddit.com{post.permalink}",
                            "upvotes": post.score,
                            "scraped_at": now_iso(),
                        }
                        records.append(rec)
                time.sleep(0.5)  # respect rate limit
            except Exception as e:
                print(f"  ✗ r/{sub_name} '{query}': {e}")
                continue

    return records


reddit_records = scrape_reddit()
append_records(reddit_records)
print(f"Reddit total: {len(reddit_records)} questions")

## 2. GitHub — Curated Interview Repos

Clones repos once, then parses markdown files for Q&A patterns.
No API key needed.

In [ ]:
import git  # gitpython

REPOS = [
    {
        "url": "https://github.com/donnemartin/system-design-primer",
        "role": "swe",
        "topic": "system_design",
    },
    {
        "url": "https://github.com/alexeygrigorev/data-science-interviews",
        "role": "ds",
        "topic": "general",
    },
    {
        "url": "https://github.com/khangich/machine-learning-interview",
        "role": "ml",
        "topic": "ml",
    },
    {
        "url": "https://github.com/yangshun/tech-interview-handbook",
        "role": "swe",
        "topic": "coding",
    },
    {
        "url": "https://github.com/ashishps1/awesome-behavioral-interviews",
        "role": "unknown",
        "topic": "behavioral",
    },
]

REPOS_DIR = OUTPUT_DIR / "repos"
REPOS_DIR.mkdir(exist_ok=True)


def clone_or_pull(repo_url: str, dest: pathlib.Path):
    if dest.exists():
        git.Repo(dest).remotes.origin.pull()
        print(f"  pulled {dest.name}")
    else:
        git.Repo.clone_from(repo_url, dest, depth=1)
        print(f"  cloned {dest.name}")


def parse_markdown_qa(md_text: str) -> list[tuple[str, str]]:
    """
    Extract Q&A pairs from markdown.
    Patterns handled:
      - Lines ending in ? (question) followed by non-empty lines (answer)
      - Q: / A: blocks
      - ## headers that are questions
    """
    pairs = []
    lines = md_text.splitlines()

    # Pattern 1: Q: ... / A: ...
    qa_block_re = re.compile(r'^[*-]?\s*[Qq][:\.]\s*(.+)')
    a_block_re  = re.compile(r'^[*-]?\s*[Aa][:\.]\s*(.+)')
    i = 0
    while i < len(lines):
        qm = qa_block_re.match(lines[i])
        if qm and i + 1 < len(lines):
            am = a_block_re.match(lines[i + 1])
            if am:
                pairs.append((qm.group(1).strip(), am.group(1).strip()))
                i += 2
                continue
        i += 1

    # Pattern 2: heading or bullet that ends in ?
    q_line_re = re.compile(r'^#{1,4}\s+(.+\?)\s*$|^[*-]\s+(.+\?)\s*$')
    for idx, line in enumerate(lines):
        m = q_line_re.match(line)
        if m:
            question = (m.group(1) or m.group(2)).strip()
            # Collect next non-empty lines as answer (up to 10)
            answer_lines = []
            for j in range(idx + 1, min(idx + 11, len(lines))):
                l = lines[j].strip()
                if l and not l.startswith("#"):
                    answer_lines.append(l)
                elif answer_lines:
                    break
            if question:
                pairs.append((question, " ".join(answer_lines)))

    return pairs


def scrape_github() -> list[dict]:
    records = []

    for repo_info in tqdm(REPOS, desc="Repos"):
        repo_name = repo_info["url"].rstrip("/").split("/")[-1]
        dest = REPOS_DIR / repo_name
        try:
            clone_or_pull(repo_info["url"], dest)
        except Exception as e:
            print(f"  ✗ clone {repo_name}: {e}")
            continue

        md_files = list(dest.rglob("*.md"))
        for md_file in md_files:
            try:
                text = md_file.read_text(encoding="utf-8", errors="ignore")
                pairs = parse_markdown_qa(text)
                for question, answer in pairs:
                    if len(question) < 15:
                        continue
                    rec = {
                        "id": make_id("github", question),
                        "source": "github",
                        "company": classify_company(question + " " + answer),
                        "role": repo_info["role"],
                        "topic": repo_info["topic"] if repo_info["topic"] != "general"
                                 else classify_topic(question),
                        "question": question,
                        "answer": answer[:3000],
                        "url": repo_info["url"] + "/blob/main/" + str(md_file.relative_to(dest)),
                        "scraped_at": now_iso(),
                    }
                    records.append(rec)
            except Exception as e:
                print(f"  ✗ parse {md_file.name}: {e}")

    return records


github_records = scrape_github()
append_records(github_records)
print(f"GitHub total: {len(github_records)} questions")

## 3. Stack Exchange API

No API key needed for read-only (throttled at 300 req/day without key, 10k with).
Get a free key at https://stackapps.com/apps/oauth/register

Targets: Stack Overflow, Cross Validated (stats/ML), Software Engineering SE.

In [ ]:
SE_API_KEY = os.getenv("SE_API_KEY", "")  # optional but recommended
SE_BASE    = "https://api.stackexchange.com/2.3"

SE_QUERIES = [
    {"site": "stackoverflow",       "tags": "python;interview-questions",  "role": "swe",  "topic": "coding"},
    {"site": "stackoverflow",       "tags": "sql;interview-questions",     "role": "ds",   "topic": "sql"},
    {"site": "stackoverflow",       "tags": "system-design;interview",     "role": "swe",  "topic": "system_design"},
    {"site": "stats",               "tags": "machine-learning;interview",  "role": "ml",   "topic": "ml"},
    {"site": "stats",               "tags": "interview-questions",         "role": "ds",   "topic": "statistics"},
    {"site": "softwareengineering", "tags": "interview-questions",         "role": "swe",  "topic": "general"},
]

SE_PAGE_SIZE = 50  # max 100
SE_PAGES     = 2   # bump to 10 for full run


def se_get(endpoint: str, params: dict) -> dict:
    if SE_API_KEY:
        params["key"] = SE_API_KEY
    params["filter"] = "withbody"
    r = requests.get(SE_BASE + endpoint, params=params, timeout=10)
    r.raise_for_status()
    return r.json()


def strip_html(text: str) -> str:
    return re.sub(r'<[^>]+>', ' ', text or "").strip()


def scrape_stackexchange() -> list[dict]:
    records = []

    for cfg in tqdm(SE_QUERIES, desc="SE sites"):
        for page in range(1, SE_PAGES + 1):
            try:
                data = se_get("/questions", {
                    "site": cfg["site"],
                    "tagged": cfg["tags"],
                    "sort": "votes",
                    "order": "desc",
                    "pagesize": SE_PAGE_SIZE,
                    "page": page,
                })
            except Exception as e:
                print(f"  ✗ SE {cfg['site']} page {page}: {e}")
                break

            for item in data.get("items", []):
                question_text = strip_html(item.get("title", ""))
                body_text     = strip_html(item.get("body", ""))
                answer_text   = ""

                # Fetch accepted answer body
                accepted_id = item.get("accepted_answer_id")
                if accepted_id:
                    try:
                        ans_data = se_get(f"/answers/{accepted_id}", {"site": cfg["site"]})
                        ans_items = ans_data.get("items", [])
                        if ans_items:
                            answer_text = strip_html(ans_items[0].get("body", ""))[:3000]
                        time.sleep(0.2)
                    except Exception:
                        pass

                if not question_text:
                    continue

                rec = {
                    "id": make_id("se", question_text),
                    "source": "stackexchange",
                    "company": classify_company(question_text + " " + body_text),
                    "role": cfg["role"],
                    "topic": cfg["topic"] if cfg["topic"] != "general"
                             else classify_topic(question_text),
                    "question": question_text,
                    "answer": answer_text or strip_html(body_text)[:2000],
                    "url": item.get("link", ""),
                    "score": item.get("score", 0),
                    "scraped_at": now_iso(),
                }
                records.append(rec)

            time.sleep(0.5)
            if not data.get("has_more"):
                break

    return records


se_records = scrape_stackexchange()
append_records(se_records)
print(f"Stack Exchange total: {len(se_records)} questions")

## 4. Deduplicate & Inspect

In [ ]:
import pandas as pd

records = []
with open(JSONL_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)

# Deduplicate on id (same question text → same hash)
before = len(df)
df = df.drop_duplicates(subset="id")
after = len(df)
print(f"Deduped: {before} → {after} records ({before - after} removed)")

# Save deduped
deduped_path = OUTPUT_DIR / "questions_deduped.jsonl"
with open(deduped_path, "w", encoding="utf-8") as f:
    for rec in df.to_dict(orient="records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Saved to {deduped_path}")
df.head()

In [ ]:
# Distribution breakdown
print("=== By source ===")
print(df["source"].value_counts().to_string())

print("\n=== By role ===")
print(df["role"].value_counts().to_string())

print("\n=== By topic ===")
print(df["topic"].value_counts().to_string())

print("\n=== By company (top 15) ===")
print(df["company"].value_counts().head(15).to_string())

print(f"\n=== Has answer ===")
has_answer = df["answer"].str.len() > 50
print(f"{has_answer.sum()} / {len(df)} records have answers")

## 5. Export for Backend Seed

Splits into per-role JSON files ready to load into Neo4j or serve as static seed data.

In [ ]:
SEED_DIR = OUTPUT_DIR / "seed"
SEED_DIR.mkdir(exist_ok=True)

for role in df["role"].unique():
    subset = df[df["role"] == role].to_dict(orient="records")
    out_path = SEED_DIR / f"{role}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(subset, f, ensure_ascii=False, indent=2)
    print(f"  {role}: {len(subset)} questions → {out_path}")

# Also one full export
full_path = SEED_DIR / "all.json"
df.to_json(full_path, orient="records", force_ascii=False, indent=2)
print(f"\nFull export: {full_path}")

## 6. Seed Kaggle Data → Neo4j

Loads the Kaggle dataset, samples 50 questions per (role × category) combo,
and writes them as `PreloadedQuestion` nodes in Neo4j.

Idempotent — re-running uses `MERGE` so no duplicates.

```
neo4j node: PreloadedQuestion {id, text, role_id, category_id, difficulty, experience, ideal_answer, keywords, source}
index:       (role_id, category_id)
```

In [ ]:
# !pip install neo4j pandas

from neo4j import GraphDatabase
import pandas as pd
import hashlib
import os

NEO4J_URI      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER",     "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

KAGGLE_CACHE = (
    "/Users/massacoulibaly/.cache/kagglehub/datasets/"
    "aryan208/hr-interview-questions-and-ideal-answers/"
    "versions/1/hr_interview_questions_dataset.json"
)

SAMPLE_PER_COMBO = 50   # questions per (role, category) combo — raise for bigger pool
BATCH_SIZE       = 500  # Neo4j write batch size

ROLE_MAP = {
    "Software Engineer":   "sde",
    "Product Manager":     "pm",
    "Data Scientist":      "ds",
    "DevOps Engineer":     "devops",
    "Marketing Associate": "marketing",
    "QA Analyst":          "qa",
    "UX Designer":         "ux",
    "HR Specialist":       "hr",
}

CATEGORY_MAP = {
    "Adaptability":       "adaptability",
    "Career Goals":       "career_goals",
    "Conflict Resolution":"conflict_resolution",
    "Culture Fit":        "culture_fit",
    "Leadership":         "leadership",
    "Motivation":         "motivation",
    "Team Collaboration": "team_collaboration",
    "Work Style":         "work_style",
}

print(f"Roles:      {list(ROLE_MAP.values())}")
print(f"Categories: {list(CATEGORY_MAP.values())}")
print(f"Max nodes:  {len(ROLE_MAP) * len(CATEGORY_MAP) * SAMPLE_PER_COMBO:,}")

In [ ]:
print("Loading Kaggle dataset...")
df_raw = pd.read_json(KAGGLE_CACHE)
print(f"Raw records: {len(df_raw):,}")

df_raw["role_id"]     = df_raw["role"].map(ROLE_MAP)
df_raw["category_id"] = df_raw["category"].map(CATEGORY_MAP)
df_raw = df_raw.dropna(subset=["role_id", "category_id"])

# Sample N per (role, category) combo — stratified
df_sample = (
    df_raw
    .groupby(["role_id", "category_id"], group_keys=False)
    .apply(lambda g: g.sample(min(SAMPLE_PER_COMBO, len(g)), random_state=42))
    .reset_index(drop=True)
)
print(f"Sampled:     {len(df_sample):,} records ({SAMPLE_PER_COMBO} per combo)")

# Preview combo counts
print("\nCombos:")
print(df_sample.groupby(["role_id", "category_id"]).size().to_string())

In [ ]:
def _make_pq_id(question: str, role_id: str, category_id: str) -> str:
    key = f"{role_id}|{category_id}|{question}"
    return "pq_" + hashlib.md5(key.encode()).hexdigest()[:12]

def _seed_batch(tx, batch):
    tx.run("""
        UNWIND $batch AS row
        MERGE (q:PreloadedQuestion {id: row.id})
        SET q.text         = row.text,
            q.role_id      = row.role_id,
            q.category_id  = row.category_id,
            q.difficulty   = row.difficulty,
            q.experience   = row.experience,
            q.ideal_answer = row.ideal_answer,
            q.keywords     = row.keywords,
            q.source       = 'kaggle'
    """, batch=batch)

records = []
for _, row in df_sample.iterrows():
    kws = row["keywords"]
    keywords_str = ",".join(kws) if isinstance(kws, list) else str(kws)
    records.append({
        "id":          _make_pq_id(row["question"], row["role_id"], row["category_id"]),
        "text":        row["question"],
        "role_id":     row["role_id"],
        "category_id": row["category_id"],
        "difficulty":  row["difficulty"],
        "experience":  row["experience"],
        "ideal_answer":row["ideal_answer"],
        "keywords":    keywords_str,
    })

print(f"Connecting to Neo4j at {NEO4J_URI}...")
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session() as session:
    # Create composite index for fast role+category lookups
    session.run("""
        CREATE INDEX preloaded_role_cat IF NOT EXISTS
        FOR (q:PreloadedQuestion) ON (q.role_id, q.category_id)
    """)
    print("Index ensured.")

    total = len(records)
    for start in range(0, total, BATCH_SIZE):
        batch = records[start:start + BATCH_SIZE]
        session.execute_write(_seed_batch, batch)
        print(f"  seeded {min(start + BATCH_SIZE, total):,} / {total:,}", end="\r")

driver.close()
print(f"\nDone. {total:,} PreloadedQuestion nodes written.")

In [ ]:
# Verify — counts per role × category in Neo4j
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as session:
    result = session.run("""
        MATCH (q:PreloadedQuestion)
        RETURN q.role_id AS role, q.category_id AS category, count(*) AS cnt
        ORDER BY role, category
    """)
    rows = result.data()

driver.close()

print(f"{'Role':15} | {'Category':25} | Count")
print("-" * 55)
for r in rows:
    print(f"{r['role']:15} | {r['category']:25} | {r['cnt']}")
print(f"\nTotal: {sum(r['cnt'] for r in rows):,} nodes in Neo4j")